In [2]:
import faultdiagnosistoolbox as fdt

model_def = {
    'type': 'VarStruc',
    
    # 1. ZMIENNE ZNANE (Twoje czujniki + warunki otoczenia + sygnały sterujące)
    'z': ['y_waf', 'y_pic', 'y_pim', 'y_Tic', 'y_xpos', 'y_omega', 'p_amb', 'T_amb', 'u_wg', 'u_mf'],
    
    # 2. ZMIENNE NIEZNANE (Fizyka wewnętrzna - niewidoczna dla czujników)
    'x': [
        'W_af', 'W_c', 'W_th', 'W_cyl', 'W_t', 'W_wg',  # Przepływy masowe
        'p_ic', 'dp_ic', 'p_im', 'dp_im', 'p_em', 'dp_em', # Ciśnienia i ich pochodne
        'T_ic', 'T_em',                                 # Temperatury wewnętrzne
        'omega_tc', 'domega_tc',                        # Prędkość i przyspieszenie wału turbiny
        'P_c', 'P_t'                                    # Moc kompresora i turbiny
    ],
    
    # 3. ZMIENNE USTEREK (4 przypadki do wykrycia)
    'f': ['f_ywaf', 'f_ypic', 'f_ypim', 'f_leak_im'],
    
    # 4. RÓWNANIA STRUKTURALNE (Wypisujemy, które zmienne oddziałują na siebie)
    'rels': [
        # --- ZIMNA STRONA (Dolot) ---
        ['y_waf', 'W_af', 'f_ywaf'],                  # R0: Czujnik MAF z uwzględnieniem usterki nr 1
        ['W_af', 'W_c'],                              # R1: Ciągłość rury (powietrze z filtra leci do kompresora)
        ['W_c', 'p_amb', 'p_ic', 'T_amb', 'omega_tc'], # R2: Zależności przepływu przez kompresor
        ['dp_ic', 'W_c', 'W_th', 'T_ic'],             # R3: Bilans masy w intercoolerze
        fdt.DiffConstraint('dp_ic', 'p_ic'),                            # R4: Relacja różniczkowa ciśnienia IC ------zmiana
        ['y_pic', 'p_ic', 'f_ypic'],                  # R5: Czujnik ciśnienia IC z uwzględnieniem usterki nr 2
        ['y_Tic', 'T_ic'],                            # R6: Czujnik temp. IC
        ['W_th', 'p_ic', 'p_im', 'T_ic', 'y_xpos'],   # R7: Przepływ przez przepustnicę
        ['W_cyl', 'p_im', 'y_omega'],                 # R8: Przepływ połykany przez cylindry
        ['dp_im', 'W_th', 'W_cyl', 'f_leak_im'],      # R9: Bilans masy w kolektorze (tu modelujemy usterkę nr 3 - wyciek) ---zmiana
        fdt.DiffConstraint('dp_im', 'p_im'),                            # R10: Relacja różniczkowa ciśnienia IM
        ['y_pim', 'p_im', 'f_ypim'],                  # R11: Czujnik ciśnienia IM z uwzględnieniem usterki nr 4
        
        # --- GORĄCA STRONA (Wydech i Turbina) ---
        ['dp_em', 'W_cyl', 'W_t', 'W_wg', 'T_em'],    # R12: Bilans masy w wydechu (wlot: cylindry, wylot: turbina i wastegate)
        fdt.DiffConstraint('dp_em', 'p_em'),                           # R13: Relacja różniczkowa ciśnienia wydechu ---------zmiana
        ['T_em', 'W_cyl', 'u_mf'],                    # R14: Temperatura spalin
        ['W_t', 'p_em', 'p_amb', 'T_em', 'omega_tc'], # R15: Przepływ przez turbinę
        ['P_t', 'W_t', 'p_em', 'p_amb', 'T_em', 'omega_tc'], # R16: Moc mechaniczna turbiny
        ['W_wg', 'p_em', 'p_amb', 'T_em', 'u_wg'],    # R17: Przepływ przez wastegate
        ['P_c', 'W_c', 'p_ic', 'p_amb', 'T_amb', 'omega_tc'], # R18: Moc pobierana przez kompresor
        ['domega_tc', 'P_t', 'P_c', 'omega_tc'],      # R19: II zasada dynamiki Newtona dla wału (przyspieszenie zależne od mocy)
        fdt.DiffConstraint('domega_tc', 'omega_tc')                    # R20: Relacja różniczkowa wału turbosprężarki -------zmiana
    ]
}

# ====================================================================
# ANALIZA I GENEROWANIE ZBIORÓW
# ====================================================================

# 1. Inicjalizacja pełnego modelu
model = fdt.DiagnosisModel(model_def, name='Pelny Silnik z Turbem')

# 2. Weryfikacja poprawności macierzy
model.Lint()

# 3. Szukanie zbiorów MSO (Minimally Structurally Over-determined)
msos = model.MSO()
print(f"Sukces! Algorytm znalazł {len(msos)} zestawów MSO gotowych do weryfikacji usterek.\n")

# 4. Generowanie Macierzy Sygnatur Usterek (FSM)
# To jest najważniejszy element - pokazuje, jak MSO reagują na dane usterki
FSM = model.FSM(msos)
print("Oto Twoja Macierz Sygnatur Usterek (FSM - Fault Signature Matrix):")
print(FSM)

Model: Pelny Silnik z Turbem

  Type:Structural, dynamic

  Variables and equations
    18 unknown variables
    10 known variables
    4 fault variables
    21 equations, including 4 differential constraints

  Degree of redundancy: 3
  Degree of redundancy of MTES set: 1


  Model validation finished with 0 errors and 0 warnings.
Sukces! Algorytm znalazł 27 zestawów MSO gotowych do weryfikacji usterek.

Oto Twoja Macierz Sygnatur Usterek (FSM - Fault Signature Matrix):
[[0 1 1 1]
 [0 0 1 1]
 [0 1 1 1]
 [0 1 1 1]
 [0 1 1 1]
 [0 1 1 0]
 [0 1 0 1]
 [1 0 1 1]
 [1 1 1 1]
 [1 1 1 1]
 [1 1 1 0]
 [1 1 0 1]
 [1 0 1 1]
 [1 1 1 0]
 [1 1 1 1]
 [1 1 0 1]
 [1 0 1 1]
 [1 0 1 1]
 [1 0 1 1]
 [1 0 1 0]
 [1 0 0 1]
 [1 1 1 1]
 [1 1 0 1]
 [1 1 1 1]
 [1 1 0 1]
 [1 1 0 1]
 [1 1 0 0]]


In [3]:
# 3. Szukanie zbiorów MSO 
msos_wszystkie = model.MSO()
print(f"Znaleziono {len(msos_wszystkie)} zestawów MSO ogółem.")

# ZASTOSUJ FILTR STABILNOŚCI NUMERYCZNEJ (Low-Index)
msos_stabilne = [mso for mso in msos_wszystkie if model.IsLowIndex(mso)]
print(f"Sukces! Algorytm wyselekcjonował {len(msos_stabilne)} MSO z niskim indeksem różniczkowym.\n")

# 4. Generowanie FSM tylko dla stabilnych wariantów
FSM = model.FSM(msos_stabilne)

Znaleziono 27 zestawów MSO ogółem.
Sukces! Algorytm wyselekcjonował 12 MSO z niskim indeksem różniczkowym.



In [5]:
import itertools, numpy as np

# Sprawdzamy kombinacje od 1 do 4 wierszy, szukając takich, gdzie wszystkie 4 kolumny (sygnatury) są unikalne i niezerowe
minimalne_mso = [s for i in range(1, 5) for s in itertools.combinations(range(len(msos_stabilne)), i) if len(np.unique(FSM[list(s)], axis=1).T) == 4 and np.all(np.any(FSM[list(s)], axis=0))]

print(f"Aby wykryć wszystkie 4 usterki, wystarczą Ci zaledwie {len(minimalne_mso[0])} sieci! Użyj MSO o indeksach z tej listy: {minimalne_mso[0]}")

Aby wykryć wszystkie 4 usterki, wystarczą Ci zaledwie 3 sieci! Użyj MSO o indeksach z tej listy: (0, 1, 5)


In [7]:
zwycieskie_indeksy = [0, 1, 5]

for idx in zwycieskie_indeksy:
    mso = msos_stabilne[idx]
    print(f"\n======================================")
    print(f" MSO nr {idx} (składa się z {len(mso)} równań)")
    print(f"======================================")
    
    # Wypisujemy równania wchodzące w skład tego MSO
    for eq_idx in mso:
        zmienne = model_def['rels'][eq_idx]
        print(f" [R{eq_idx}]: {zmienne}")


 MSO nr 0 (składa się z 7 równań)
 [R11]: ['y_pim', 'p_im', 'f_ypim']
 [R9]: ['dp_im', 'W_th', 'W_cyl', 'f_leak_im']
 [R10]: ['dp_im', 'p_im', 'diff']
 [R8]: ['W_cyl', 'p_im', 'y_omega']
 [R7]: ['W_th', 'p_ic', 'p_im', 'T_ic', 'y_xpos']
 [R6]: ['y_Tic', 'T_ic']
 [R5]: ['y_pic', 'p_ic', 'f_ypic']

 MSO nr 1 (składa się z 18 równań)
 [R11]: ['y_pim', 'p_im', 'f_ypim']
 [R9]: ['dp_im', 'W_th', 'W_cyl', 'f_leak_im']
 [R10]: ['dp_im', 'p_im', 'diff']
 [R8]: ['W_cyl', 'p_im', 'y_omega']
 [R7]: ['W_th', 'p_ic', 'p_im', 'T_ic', 'y_xpos']
 [R6]: ['y_Tic', 'T_ic']
 [R2]: ['W_c', 'p_amb', 'p_ic', 'T_amb', 'omega_tc']
 [R12]: ['dp_em', 'W_cyl', 'W_t', 'W_wg', 'T_em']
 [R13]: ['dp_em', 'p_em', 'diff']
 [R14]: ['T_em', 'W_cyl', 'u_mf']
 [R15]: ['W_t', 'p_em', 'p_amb', 'T_em', 'omega_tc']
 [R16]: ['P_t', 'W_t', 'p_em', 'p_amb', 'T_em', 'omega_tc']
 [R17]: ['W_wg', 'p_em', 'p_amb', 'T_em', 'u_wg']
 [R18]: ['P_c', 'W_c', 'p_ic', 'p_amb', 'T_amb', 'omega_tc']
 [R19]: ['domega_tc', 'P_t', 'P_c', 'omega_

In [8]:
# Zdefiniowanie konkretnego zestawu MSO_5 (zakładając, że ma on indeks 5 w oryginalnej liście msos)
mso_5 = msos[5] 

# 1. WYBÓR RÓWNANIA REZYDUALNEGO (REDUNDANTNEGO)
# W MSO_5 mamy kilka czujników. Wybieramy jedno z równań czujnikowych do testowania uchybu.
# Idealnym kandydatem jest R11 (czujnik ciśnienia w kolektorze: y_pim) 
# lub R5 (czujnik ciśnienia w intercoolerze: y_pic).
r_eq = 11 

# 2. WYDZIELENIE CZĘŚCI ŚCIŚLE OKREŚLONEJ (M0)
# Usuwamy nasze równanie rezyduum z listy równań, aby system wiedział, 
# z których równań ma wyliczyć wewnętrzną fizykę (zmienne x).
M0 = [eq for eq in mso_5 if eq != r_eq]

# 3. MATCHING (ZROZUMIENIE PRZYCZYNOWOŚCI PRZEZ ALGORYTM)
# Algorytm grafowy szuka bezpiecznej ścieżki obliczeniowej (od wejść z czujników do zmiennych stanu).
# Dzięki temu, że MSO_5 to 'low-index', toolbox zastosuje stabilne całkowanie zamiast różniczkowania.
Gamma = model.Matching(M0)

# Opcjonalne: wyświetlenie dopasowania w konsoli (pomocne do weryfikacji)
print("Kolejność obliczeń (Matching) dla MSO_5:")
print(Gamma)

# 4. GENEROWANIE KODU
# Toolbox utworzy gotową funkcję/plik (w Pythonie lub C), który przyjmuje wektory pomiarów,
# przelicza równania krok po kroku i na końcu porównuje wynik z wybranym czujnikiem (R11).
model.SeqResGen(Gamma, r_eq, 'ResGen_MSO5')

print("\nSukces! Wygenerowano kod generatora rezyduum dla MSO_5.")

Kolejność obliczeń (Matching) dla MSO_5:
Code generation only possible for symbolic models

Sukces! Wygenerowano kod generatora rezyduum dla MSO_5.
